In [1]:
%pip install dotenv
%pip install pypdf2
%pip install pdfrw
%pip install reportlab
%pip install --upgrade pypdf2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dotenv]

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.6 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restar

In [1]:
import os
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
import pandas as pd
from io import StringIO

# Load environment variables from environment.env
load_dotenv("environment.env")

# Get values from env
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
template_container_name = os.getenv("TEMPLATE_CONTAINER_NAME")

# Initialize client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(template_container_name)

# Example: List blobs
print("Blobs in container:")
for blob in container_client.list_blobs():
    print("  -", blob.name)


Blobs in container:
  - ADA Notice to Borrowers-FORM.pdf
  - Acknowledgement of Receipt - FORM.pdf
  - Agreement to Cooperate-FORM.pdf
  - Condominium Rider-FORM.pdf
  - Disclosure Statement-FORM.pdf
  - Escrow Waiver Affidavit-FORM.pdf
  - Info Disclosure Author-FORM.pdf
  - Mortgage-FORM.pdf
  - Perjury Statement-FORM.pdf
  - Primary Residence Affidavit-FORM.pdf
  - Promissory Note-FORM.pdf
  - SSICollect-FORM.pdf


#### Fields not annotated on the fillable pdfs

In [3]:
import os
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
from PyPDF2 import PdfReader
from io import BytesIO

# Load env file
load_dotenv("environment.env")
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
template_container_name = os.getenv("TEMPLATE_CONTAINER_NAME")

# Connect to blob
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(template_container_name)

# Iterate through all PDFs in the container
for blob in container_client.list_blobs():
    if blob.name.lower().endswith(".pdf"):
        print(f"\n📄 Processing PDF: {blob.name}")
        
        # Download PDF into memory
        blob_client = container_client.get_blob_client(blob.name)
        pdf_bytes = blob_client.download_blob().readall()
        pdf_stream = BytesIO(pdf_bytes)

        # Read PDF
        reader = PdfReader(pdf_stream)

        # Embark on reading page by page
        for i, page in enumerate(reader.pages):
            print(f"\n--- Page {i+1} ---")
        
            if "/Annots" in page:
                for annot in page["/Annots"]:
                    annot_obj = annot.get_object()
                    if annot_obj.get("/Subtype") == "/Widget":  # Widget = form field
                        field = annot_obj.get("/T")   # Field name
                        value = annot_obj.get("/V")   # Field value
                        print(f"Field: {field}, Value: {value}")
            else:
                print("No form fields on this page.")



📄 Processing PDF: ADA Notice to Borrowers-FORM.pdf

--- Page 1 ---
Field: BORROWER, Value: None
Field: COBORROWER, Value: None
Field: DATE, Value: None
Field: DATE_2, Value: None

📄 Processing PDF: Acknowledgement of Receipt - FORM.pdf

--- Page 1 ---
Field: borrowers_line, Value: None
Field: Date, Value: None
Field: Loan Number, Value: None
Field: Property Address, Value: None
Field: Picture ID for All Borrowers, Value: None
Field: Promissory Note, Value: None
Field: Mortgage, Value: None
Field: Condo Rider, Value: None
Field: TruthInLending Disclosure Statement, Value: None
Field: Right of Rescission 2 copies for the borrower, Value: None
Field: Escrow Disclosure Loan Proceeds to Be Put in Escrow, Value: None
Field: Primary Residence Affidavit, Value: None
Field: Agreement to Cooperate, Value: None
Field: Disclosure Statement, Value: None
Field: Information Authorization Disclosure, Value: None
Field: Perjury Statement, Value: None
Field: SSN Policy, Value: None
Field: ADA Notice to